In [5]:
import pandas as pd

df = pd.read_csv("C:/Users/mengw/OneDrive/Documents/Profesional Work/DOTA2_Esport_Predictor/data/processed/features.csv")

# Optional: clean column names
df.columns = df.columns.str.strip()

# Bucket synergy into groups
df["synergy_bucket"] = pd.cut(
    df["synergy_score"],
    bins=[-1, 0.2, 0.5, 1, 10],
    labels=["low", "medium", "high", "very_high"]
)

# Check win rate per bucket
result = df.groupby("synergy_bucket")["win"].mean()

print(result)

synergy_bucket
low          0.512500
medium       0.000000
high         0.533333
very_high    0.347826
Name: win, dtype: float64


C:\Users\mengw\AppData\Local\Temp\ipykernel_20820\4107883426.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  result = df.groupby("synergy_bucket")["win"].mean()


In [6]:
print(df[["synergy_score", "win"]].corr())

               synergy_score       win
synergy_score       1.000000 -0.008362
win                -0.008362  1.000000


In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier

INPUT_PATH = "C:/Users/mengw/OneDrive/Documents/Profesional Work/DOTA2_Esport_Predictor/data/processed/features.csv"


def load_data():
    df = pd.read_csv(INPUT_PATH)

    # Clean column names
    df.columns = df.columns.str.strip()

    return df


def prepare_features(df):
    # Features to use
    features = [
        "total_kills",
        "total_deaths",
        "avg_gpm",
        "avg_xpm",
        "synergy_score",
        "activity",
        "recent_winrate"
    ]

    X = df[features]
    y = df["win"]

    return X, y


def train_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    return model


def feature_importance(model, X):
    importances = model.feature_importances_

    for name, score in zip(X.columns, importances):
        print(f"{name}: {score:.4f}")


def main():
    df = load_data()

    # Optional: remove cold-start rows
    # df = df[df["activity"] > 0]

    X, y = prepare_features(df)

    model = train_model(X, y)

    print("\nFeature Importance:")
    feature_importance(model, X)


if __name__ == "__main__":
    main()

Accuracy: 0.9230769230769231

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.94      0.92        18
           1       0.95      0.90      0.93        21

    accuracy                           0.92        39
   macro avg       0.92      0.92      0.92        39
weighted avg       0.92      0.92      0.92        39


Feature Importance:
total_kills: 0.1992
total_deaths: 0.2694
avg_gpm: 0.3295
avg_xpm: 0.1567
synergy_score: 0.0203
activity: 0.0158
recent_winrate: 0.0091
